# 🚀 MLOps.dev — 60-Second Edge Deployment Quickstart

Deploy, monitor, and auto-heal machine learning models on physical and virtual edge devices (Jetson, Raspberry Pi, Coral TPU).

[MLOps.dev Live Control Plane](https://mlopsde.me) | [GitHub Repository](https://github.com/Raghunath2604/Raghunath2604-mlops-dev)

### Step 1: Install the MLOps.dev Python SDK

In [ ]:
!pip install mlops-dev

### Step 2: Initialize Client & Authenticate

In [ ]:
import mlops_dev as mlops

# Connect to production control plane using demo sandbox token
client = mlops.Client(api_key="demo", base_url="https://mlopsde.me/api")
print("Connected to MLOps.dev Fleet API")

### Step 3: Create & Push a Lightweight ONNX Model

In [ ]:
# Create sample binary model artifact for demonstration
with open("defect_detector_v1.onnx", "wb") as f:
    f.write(b"ONNX_EDGE_MODEL_WEIGHTS_SAMPLE_PAYLOAD_BUFFER_123456789")

# Upload and register model version
version = client.models.push(
    "./defect_detector_v1.onnx",
    name="defect-detector",
    tag="v1.0",
    format="onnx",
    variant="jetson_orin"
)
print(f"✅ Successfully registered {version.name}:{version.tag} (SHA-256: {version.sha256[:12]}...)")

### Step 4: Query Online Fleet Hardware

In [ ]:
devices = client.devices.list(status="online")
print(f"Discovered {len(devices)} active edge hardware devices:")
for d in devices[:5]:
    print(f" - [{d.id}] {d.name} | Silicon: {d.hw_class} | Drift: {d.drift_score} | Latency: {d.latency_ms}ms")

### Step 5: Trigger Staged Canary Rollout with Health Gating

In [ ]:
dep = client.deploy(
    "defect-detector:v1.0",
    target="all",
    stages=[
        {"hw_class": "jetson_orin", "pct": 25},
        {"hw_class": "all", "pct": 100}
    ],
    health_gate={
        "accuracy_delta": -0.03,
        "latency_delta": 0.20
    }
)
print(f"🚀 Canary Rollout Dispatched: Deployment ID #{dep.id}")
print(f"Deployment Status: {dep.status}")